# 04 ML Classical Algorithms

This notebook uses `MLTrainAndStore` to train regressors with only the classical-network features.

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor, HistGradientBoostingRegressor, RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.neural_network import MLPRegressor
from sklearn.svm import SVR
import xgboost as xgb
from xgboost import XGBRegressor

from src.models.ml_train_and_store import (
    MLTrainAndStore,
    load_classical_ml_dataset,
    make_log_regression_model,
)

pd.set_option("display.max_columns", 200)

## Load Dataset

In [2]:
df, feature_cols = load_classical_ml_dataset(PROJECT_ROOT)
df.shape, feature_cols

((145536, 88),
 ['degree_centrality_in',
  'degree_centrality_out',
  'degree_centrality_total',
  'betweenness_centrality',
  'closeness_centrality',
  'weighted_degree_in',
  'weighted_degree_out',
  'weighted_degree_total',
  'pagerank',
  'debtrank'])

In [3]:
trainer = MLTrainAndStore(
    df=df,
    feature_cols=feature_cols,
    target_col="systemic_risk_label",
)

trainer.train_df.shape, trainer.val_df.shape, trainer.test_df.shape

((109152, 88), (18192, 88), (13644, 88))

## Define Models

In [4]:
candidate_models = {
    "linear_regression": make_log_regression_model(LinearRegression(), scale_features=True),
}

list(candidate_models)

['linear_regression']

## Train And Store

In [5]:
trainer.train_many(candidate_models)

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.15017,0.231617,0.163151,6.950552,4.488082,3.688349,-12.881284,-1.165472,-2.473429


In [6]:
trainer.results()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,linear_regression,0.15017,0.231617,0.163151,6.950552,4.488082,3.688349,-12.881284,-1.165472,-2.473429


## Single-Model Pattern

Use this when you want to add one model manually.

In [7]:
amodel = make_log_regression_model(
    XGBRegressor(n_estimators=100, learning_rate=0.1, max_depth=6, random_state=42),
    scale_features=True,
)

trainer.train_and_store(model=amodel, name="XGBRegressor_search_1")
trainer.results()

,model,train_mae,validation_mae,test_mae,train_rmse,validation_rmse,test_rmse,train_r2,validation_r2,test_r2
0,XGBRegressor_search_1,0.036024,0.125297,0.07132,0.43008,1.687496,0.864754,0.946852,0.693863,0.809067
1,linear_regression,0.15017,0.231617,0.163151,6.950552,4.488082,3.688349,-12.881284,-1.165472,-2.473429


## Best Model

In [8]:
trainer.best_model_name()

'XGBRegressor_search_1'

In [9]:
trainer.predict_test().head(20)

,bank_id,year,quarter,period,systemic_risk_label,prediction,abs_error
0,5,2023,1,2023Q1,72,29.082319,42.917681
1,5,2023,2,2023Q2,60,27.318760,32.681240
2,0,2023,1,2023Q1,56,27.318760,28.681240
3,8,2023,1,2023Q1,53,26.098705,26.901295
4,6,2023,1,2023Q1,42,18.846636,23.153364
5,1,2023,1,2023Q1,41,18.605961,22.394039
6,2,2023,1,2023Q1,40,19.238838,20.761162
7,7,2023,1,2023Q1,36,18.860182,17.139818
8,6,2023,2,2023Q2,36,18.956228,17.043772
9,17,2023,1,2023Q1,44,27.625662,16.374338
